# 白酒行业财报 RAG 问答知识库 · Colab 一键部署

在 Colab 上按顺序运行下面的单元格即可。全程约 20–40 分钟（主要耗在下载 PDF 和编码）。

**运行前**：菜单 `代码执行程序 → 更改运行时类型 → 硬件加速器选 GPU`（可选，CPU 也能跑，只是慢些）。

**产出**：一个公网可访问的 Streamlit 问答页面链接（cloudflared 临时隧道）。

## 0 · 检查环境（GPU 可选）

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo '未分配 GPU，将使用 CPU 运行（可接受）'
!python -V && df -h /content | tail -1

## 1 · 克隆代码仓库

把 `YOUR_NAME/baijiu-finrag` 换成你自己的 GitHub 仓库地址。

In [ ]:
REPO = "https://github.com/YOUR_NAME/baijiu-finrag.git"   # ← 改成你的仓库
%cd /content
!rm -rf baijiu-finrag
!git clone $REPO baijiu-finrag
%cd /content/baijiu-finrag
!ls -la

## 2 · 安装依赖

Colab 里 `faiss-cpu` 和 `pdfplumber` 一般自带，这里统一装一遍保证版本一致。

In [ ]:
!pip install -q -r requirements.txt
!pip install -q cloudflared 2>/dev/null; which cloudflared || (wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared)
!cloudflared --version

## 3 · 下载年报 PDF（12 家白酒公司）

走巨潮资讯网接口。若某几家失败，改用浏览器手工下载后上传到 `data/raw/<公司>_<代码>/2024_年报.pdf`（第 4 格给了上传通道）。

In [ ]:
!python src/01_download.py
!echo '--- 校验 ---' && python src/01_download.py --check

### 3b · （可选）手工补传 PDF

如果第 3 格有公司下载失败，用下面的格子上传对应 PDF，记得改文件名。

In [ ]:
from google.colab import files
import shutil, os

COMPANY_DIR = "data/raw/贵州茅台_600519"   # ← 改成目标公司目录
os.makedirs(COMPANY_DIR, exist_ok=True)
up = files.upload()                      # 选择本地的年报 PDF
for fn in up:
    shutil.move(fn, os.path.join(COMPANY_DIR, '2024_年报.pdf'))
    print('已放置 →', os.path.join(COMPANY_DIR, '2024_年报.pdf'))

## 4 · 抽取文字与表格

留意输出的自检表：**任何一家「表格块 = 0」都说明抽取异常**，需要把该 PDF 换成文本层版本。

In [ ]:
!python src/02_extract.py

## 5 · 切块（带公司 / 章节 / 页码元数据）

In [ ]:
!python src/03_chunk.py
!head -c 1200 data/chunks/chunks.jsonl

## 6 · 建向量索引 + BM25 索引

首次会下载 bge-m3 权重（约 2GB）。约 5–15 分钟。

In [ ]:
!python src/04_index.py
!ls -lh index/ index/faiss/

## 7 · 用命令行跑通一次问答（先验证检索与生成链路）

生成器默认用 `ollama`。Colab 上没有本地 ollama，二选一：

- **A. 只验证检索**：把 `config.yaml` 里 `generate.provider` 改成 `none`，或直接看这一格输出的召回片段；
- **B. 接在线 API**：填 `OPENAI_API_KEY`（或任何兼容 OpenAI 协议的服务，改 `openai_model` 与 base_url 即可）。

In [ ]:
# 方案 B：接在线 LLM（不填也能跑，只是答案区为空）
import os
# os.environ['OPENAI_API_KEY'] = 'sk-...'

# 让 config 使用 openai（若不用生成，可跳过）
import yaml
cfg = yaml.safe_load(open('config.yaml', encoding='utf-8'))
cfg['generate']['provider'] = 'openai' if os.environ.get('OPENAI_API_KEY') else 'none'
yaml.safe_dump(cfg, open('config.yaml', 'w', encoding='utf-8'), allow_unicode=True)
print('provider =', cfg['generate']['provider'])

In [ ]:
!python src/05_rag.py "2024年贵州茅台营业收入是多少？"

## 8 · 启动 Streamlit 页面并拿公网链接

下面两格先后运行。注意 cloudflared 输出的 `https://xxxx.trycloudflare.com` 就是**你截图要用的地址**（临时链接，本次会话有效）。

In [ ]:
import subprocess, time
subprocess.Popen(["streamlit", "run", "app/streamlit_app.py",
                  "--server.port", "8501", "--server.headless", "true",
                  "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
                 stdout=open('streamlit.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(12)
print(open('streamlit.log').read()[-1500:])

In [ ]:
!cloudflared tunnel --url http://localhost:8501 2>&1 | tee cloudflared.log
# 运行后看输出里的 https://xxxx.trycloudflare.com —— 那就是你的问答页面地址，用它截图

## 9 · 跑 10 道评测题

In [ ]:
!python src/06_eval.py
import glob
from IPython.display import Markdown, display
latest = sorted(glob.glob('eval/results_*.md'))[-1]
print('结果文件：', latest)
display(Markdown(open(latest, encoding='utf-8').read()[:3000]))

## 10 · （可选）把索引与结果存回 Google Drive

Colab 断开后索引会丢失，重新建要再跑一次编码。存 Drive 可以下次直接复用。

> 不要把索引提交到 GitHub —— 体积大。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/baijiu-finrag-cache
!cp -r index data/extracted data/chunks /content/drive/MyDrive/baijiu-finrag-cache/
!cp eval/results_*.md /content/drive/MyDrive/baijiu-finrag-cache/ 2>/dev/null || true
print('已备份到 Google Drive')

# 下次会话恢复：
# !cp -r /content/drive/MyDrive/baijiu-finrag-cache/index .
# !cp -r /content/drive/MyDrive/baijiu-finrag-cache/data ./data

## 交付前检查

- [ ] 12 家公司 PDF 都下到了（`python src/01_download.py --check`）
- [ ] 每家公司表格块数 > 0
- [ ] 页面能提问、答案带出处、展开能看到召回片段
- [ ] 已用 cloudflared 链接完成页面截图（放进 `docs/screenshots/`）
- [ ] 10 道题跑完，`eval/results_*.md` 里对错与错因已人工填写
- [ ] `eval/结论_一页.md` 写完了